In [1]:

import json
import sys
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))

from uplift.data import make_splits, take, TREATMENT
from uplift.learners import TLearner
from uplift.metrics import (
    qini_coefficient,
    bootstrap_metrics,
    bootstrap_ci,
    paired_difference,
)

print("Project directory:", ROOT)


Project directory: /Users/kajasaikiran/Desktop/causal-uplift-git


In [2]:

import json
import gc
import time

# Load selected features
with open(ROOT / "results" / "kept_features.json") as f:
    kept_features = json.load(f)

# Load the tuned parameters
with open(ROOT / "results" / "best_params.json") as f:
    best_params = json.load(f)

# Load only the columns needed for visit prediction
needed_columns = kept_features + ["treatment", "visit"]

print("Loading required dataset columns...", flush=True)
start = time.time()

df = pd.read_parquet(
    ROOT / "data" / "criteo.parquet",
    columns=needed_columns
)

print(f"Dataset loaded in {time.time() - start:.1f} seconds", flush=True)

# Use the same fixed splits as the project
test_ids, pool_ids = make_splits(len(df))

# Build training and holdout datasets
dev = take(df, pool_ids[:1_000_000])
hold = take(df, test_ids)

# Free unnecessary memory
del df
gc.collect()

print("Training rows:", len(dev))
print("Holdout rows:", len(hold))
print("Selected features:", kept_features)
print("Tuned parameters:", best_params["visit"])
print("Cell 2 completed successfully!")


Loading required dataset columns...
Dataset loaded in 0.2 seconds
Training rows: 1000000
Holdout rows: 2000000
Selected features: ['f0', 'f2', 'f3', 'f4', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']
Tuned parameters: {'num_leaves': 15, 'learning_rate': 0.05, 'n_estimators': 200, 'min_child_samples': 200}
Cell 2 completed successfully!


In [3]:
X_train = dev[kept_features]
y_train = dev["visit"]
t_train = dev[TREATMENT]

In [4]:

import time

# Prepare the holdout data
X_hold = hold[kept_features]
y_hold = hold["visit"].to_numpy()
t_hold = hold[TREATMENT].to_numpy()

# Model 1: Tuned T-Learner
print("Training tuned T-Learner...", flush=True)
start = time.time()

tuned_model = TLearner(best_params["visit"])
tuned_model.fit(X_train, t_train, y_train)
tuned_scores = tuned_model.predict_uplift(X_hold)

print(f"Tuned model finished in {time.time() - start:.1f} seconds")

# Model 2: Untuned T-Learner
print("Training untuned T-Learner...", flush=True)
start = time.time()

untuned_model = TLearner()
untuned_model.fit(X_train, t_train, y_train)
untuned_scores = untuned_model.predict_uplift(X_hold)

print(f"Untuned model finished in {time.time() - start:.1f} seconds")
print("Both T-Learners trained and scored successfully!")


Training tuned T-Learner...
Tuned model finished in 4.7 seconds
Training untuned T-Learner...
Untuned model finished in 10.0 seconds
Both T-Learners trained and scored successfully!


In [5]:

# Calculate the Qini coefficient for each model
# Both models are evaluated on the same holdout dataset

qini_tuned = qini_coefficient(
    y_hold,
    t_hold,
    tuned_scores
)

qini_untuned = qini_coefficient(
    y_hold,
    t_hold,
    untuned_scores
)

# Calculate the difference
qini_difference = qini_tuned - qini_untuned

print("Tuning Sensitivity Results")
print("--------------------------")

print(f"Tuned T-Learner Qini:   {qini_tuned:.6f}")
print(f"Untuned T-Learner Qini: {qini_untuned:.6f}")
print(f"Difference:             {qini_difference:.6f}")


Tuning Sensitivity Results
--------------------------
Tuned T-Learner Qini:   0.003508
Untuned T-Learner Qini: 0.003128
Difference:             0.000380


In [6]:

# Bootstrap analysis for tuned and untuned T-Learners

print("Starting bootstrap analysis...", flush=True)

# Evaluate both models on the same 200 bootstrap samples
boot = bootstrap_metrics(
    y_hold,
    t_hold,
    {
        "tuned": tuned_scores,
        "untuned": untuned_scores
    },
    n_boot=200,
    seed=42,
    n_jobs=2
)

# Calculate the 95% confidence intervals
ci = bootstrap_ci(boot)

# Calculate the paired difference
difference = paired_difference(
    boot,
    "tuned",
    "untuned",
    qini_tuned,
    qini_untuned
)

print("\n95% Confidence Intervals:")
print(ci)

print("\nPaired Difference:")
print(difference)

print("\nBootstrap analysis completed!")


Starting bootstrap analysis...

95% Confidence Intervals:
               lo        hi
tuned    0.003194  0.003840
untuned  0.002828  0.003447

Paired Difference:
{'diff': 0.000380377574999341, 'lo': 0.0001604579986055928, 'hi': 0.0005970230558096065, 'p_one_sided': 0.004975124378109453}

Bootstrap analysis completed!


In [7]:

# Save the tuning sensitivity results

results = pd.DataFrame([
    {
        "setting": "tuned",
        "qini_coef": qini_tuned,
        "ci_lo": ci.loc["tuned", "lo"],
        "ci_hi": ci.loc["tuned", "hi"]
    },
    {
        "setting": "untuned",
        "qini_coef": qini_untuned,
        "ci_lo": ci.loc["untuned", "lo"],
        "ci_hi": ci.loc["untuned", "hi"]
    },
    {
        "setting": "difference",
        "qini_coef": difference["diff"],
        "ci_lo": difference["lo"],
        "ci_hi": difference["hi"]
    }
])

# Save CSV in the project's results folder
output_path = ROOT / "results" / "sensitivity_tuning_visit.csv"
results.to_csv(output_path, index=False)

print("Final S4 Results:")
display(results)

print("Saved to:", output_path)

# Verify assignment requirements
assert results.shape == (3, 4)
assert list(results.columns) == [
    "setting", "qini_coef", "ci_lo", "ci_hi"
]

print("CSV structure verified successfully!")

# Conclusion based on the actual confidence interval
if difference["lo"] > 0:
    conclusion = "Tuning improved the T-Learner Qini coefficient."
elif difference["hi"] < 0:
    conclusion = "Tuning reduced the T-Learner Qini coefficient."
else:
    conclusion = "There is no clear evidence that tuning changed the Qini coefficient."

print("\nConclusion:", conclusion)


Final S4 Results:


,setting,qini_coef,ci_lo,ci_hi
0,tuned,0.003508,0.003194,0.003840
1,untuned,0.003128,0.002828,0.003447
2,difference,0.000380,0.000160,0.000597


Saved to: /Users/kajasaikiran/Desktop/causal-uplift-git/results/sensitivity_tuning_visit.csv
CSV structure verified successfully!

Conclusion: Tuning improved the T-Learner Qini coefficient.


In [8]:

# Create the conclusion using the calculated results

if difference["lo"] > 0:
    finding = "Tuning improved the Qini coefficient."
elif difference["hi"] < 0:
    finding = "Tuning reduced the Qini coefficient."
else:
    finding = "The difference was not statistically clear."

conclusion = (
    f"The tuned T-Learner achieved a Qini coefficient of "
    f"{qini_tuned:.6f}, compared with {qini_untuned:.6f} "
    f"for the untuned model. The difference was "
    f"{difference['diff']:.6f}, with a 95% confidence interval "
    f"from {difference['lo']:.6f} to {difference['hi']:.6f}. "
    f"{finding}"
)

display(Markdown(f"### Conclusion\n\n{conclusion}"))


### Conclusion

The tuned T-Learner achieved a Qini coefficient of 0.003508, compared with 0.003128 for the untuned model. The difference was 0.000380, with a 95% confidence interval from 0.000160 to 0.000597. Tuning improved the Qini coefficient.